In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))  # If GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. Initialize ResNet50 model
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4)  # 4 classes for OCT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# 2. Define dual optimizers using SGD and Adam
#optimizer_SGD_1 = optim.SGD(model.parameters(), lr=0.001)
optimizer_new = optim.Adam(model.parameters(), lr=0.001)
criterion_new = nn.CrossEntropyLoss()

In [ ]:
# 3. Load Retinal OCT Images dataset
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # ResNet expects 3 channels
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
dataset = datasets.ImageFolder(root=r"C:\Users\NAGARAJUPALLY\Projects\classification\OCT2017\train - Copy", transform=transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

val_data=datasets.ImageFolder(root=r"C:\Users\NAGARAJUPALLY\Projects\classification\OCT2017\val - Copy", transform=transform)
val_loader = DataLoader(val_data, batch_size=16, shuffle=True)
# Print dataset details
print(f"Real samples: {len(dataset)}")
print("Classes:", dataset.classes)

print(f"Validation samples: {len(val_data)}")
print("Classes:", dataset.classes)

In [ ]:
import matplotlib.pyplot as plt
import torch

num_epochs = 50
train_loss_list = []
train_acc_list  = []
val_loss_list   = []
val_acc_list    = []

for epoch in range(num_epochs):
    # ===== Train =====
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer_new.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion_new(outputs, labels)
        loss.backward()
        optimizer_new.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc  = 100.0 * correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # ===== Validate =====
    model.eval()
    val_running_loss = 0.0
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion_new(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc  = 100.0 * val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% "
          f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# ===== Plots =====
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curve'); plt.legend()

plt.subplot(1,2,2)
plt.plot(train_acc_list, label='Train Acc')
plt.plot(val_acc_list,   label='Val Acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title('Accuracy Curve'); plt.legend()
plt.show()

# Optional: save checkpoint
torch.save(model.state_dict(), "resnet50-new-realdru-nor.pth")
print("Saved: resnet50-new-realdru-nor.pth")

In [ ]:
%pip install -q scikit-learn


In [ ]:
 pip install --upgrade pip

In [ ]:

from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
test_data = datasets.ImageFolder(root=r"C:\Users\NAGARAJUPALLY\Projects\classification\OCT2017\test - Copy", transform=test_transform)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

# Load model weights
model.load_state_dict(torch.load("resnet50-new-realdru-nor.pth", map_location=device))
model.eval()

correct, total = 0, 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Metrics
test_acc = 100 * correct / total
f1 = f1_score(all_labels, all_preds, average='weighted')
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_data.classes)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.savefig("newreal_confusion.eps", format='eps', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. Initialize ResNet50 model
model2 = models.resnet50(weights=None)
model2.fc = nn.Linear(model.fc.in_features, 4)  # 4 classes for OCT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2 = model.to(device)
optimizer2 = optim.Adam(model.parameters(), lr=0.001)
criterion2 = nn.CrossEntropyLoss()

In [ ]:
# 3. Load Retinal OCT Images dataset
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # ResNet expects 3 channels
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
dataset2 = datasets.ImageFolder(root=r"C:\Users\NAGARAJUPALLY\Projects\classification\OCT2017\train - Copy", transform=transform)
train_loader2 = DataLoader(dataset2, batch_size=16, shuffle=True)

val_data2=datasets.ImageFolder(root=r"C:\Users\NAGARAJUPALLY\Projects\classification\OCT2017\val - Copy", transform=transform)
val_loader2 = DataLoader(val_data2, batch_size=16, shuffle=True)
# Print dataset details
print(f"Real samples: {len(dataset2)}")
print("Classes:", dataset2.classes)

print(f"Validation samples: {len(val_data2)}")
print("Classes:", dataset2.classes)

In [ ]:
import matplotlib.pyplot as plt
import torch

num_epochs = 50
train_loss_list = []
train_acc_list  = []
val_loss_list   = []
val_acc_list    = []

for epoch in range(num_epochs):
    # ===== Train =====
    model2.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader2:
        images, labels = images.to(device), labels.to(device)

        optimizer2.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion2(outputs, labels)
        loss.backward()
        optimizer2.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader2.dataset2)
    train_acc  = 100.0 * correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # ===== Validate =====
    model2.eval()
    val_running_loss = 0.0
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion_new(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc  = 100.0 * val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% "
          f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# ===== Plots =====
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curve'); plt.legend()

plt.subplot(1,2,2)
plt.plot(train_acc_list, label='Train Acc')
plt.plot(val_acc_list,   label='Val Acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title('Accuracy Curve'); plt.legend()
plt.show()

# Optional: save checkpoint
torch.save(model.state_dict(), "resnet50-new-realdru-nor.pth")
print("Saved: resnet50-new-realdru-nor.pth")

In [ ]:

from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
test_data = datasets.ImageFolder(root=r"C:\Users\NAGARAJUPALLY\Projects\classification\OCT2017\test - Copy", transform=test_transform)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

# Load model weights
model2.load_state_dict(torch.load("resnet50-new-realdru-nor.pth", map_location=device))
model2.eval()

correct, total = 0, 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Metrics
test_acc = 100 * correct / total
f1 = f1_score(all_labels, all_preds, average='weighted')
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_data.classes)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.savefig("newgen_confusion.eps", format='eps', dpi=300, bbox_inches='tight')
plt.show()